> 📓 **Lesson 1.8 — Part 3 of 4: Data Transformation**
>
> This notebook was split out of the original single `eda_basic.ipynb` so each part can be opened and run on its own. If you are starting here rather than at Part 1, run the **Setup** cell below first — it loads the same data used throughout Lesson 1.8.
>
> Other notebooks in this set: `Part_1_descriptive_statistics.ipynb`, `Part_2_data_quality.ipynb`, `Part_4_reading_writing_data.ipynb`

# Lesson 1.8: EDA Basic

Welcome to Exploratory Data Analysis. This notebook takes one raw, messy business file and walks the
full path: understanding its structure, cleaning it, transforming it, and moving it in and out of
files.

**Structure — the four learning outcomes, in order:**
* **Part 1: Descriptive Statistics** — *summarise* a dataset: shape, data types, distributions.
* **Part 2: Data Quality** — *handle* the messy reality: missing values, duplicates, impossible values.
* **Part 3: Data Transformation** — *transform* for analysis: mapping, labels, strings, categories, dates.
* **Part 4: Reading & Writing Data** — *read and write* CSV, JSON, Excel, databases.

**For Learners:** read the `# 👉` comment above each line before you run the cell. The comment says
what the line does in plain English; the output shows you it happened.


> **🧭 Today's flow — 150 minutes.** One messy file, four learning outcomes, in order:
>
> | | Section | Learning outcome | Time |
> |---|---|---|---|
> | — | Setup + why this matters | | 5 min |
> | **Part 1** | Descriptive Statistics | **Summarise** a dataset: shape, dtypes, distributions | 33 min |
> | ☕ | *Break* | | 10 min |
> | **Part 2** | Data Quality | **Handle** missing values, duplicates, impossible values | 42 min |
> | ☕ | *Break* | | 10 min |
> | **Part 3** | Data Transformation | **Transform**: mapping, labels, strings, categories, dates, grouping | 35 min |
> | **Part 4** | Reading & Writing Data | **Read and write** CSV, JSON, Excel, databases | 15 min |
>
> **The spine:** we work on one file, `data/cafe_june_raw.csv`, from start to finish. Each section
> improves the same `clean` table, and Part 4 saves it. Small hand-built tables appear alongside it
> as *drills* — they isolate one method so you can see exactly what it does.
>
> Each of Parts 1–3 ends with a **🛠️ Group Exercise**. Deep dives live in `reference.md`;
> the Appendix at the end is self-study.


### The business problem

> **The Daily Grind** is a four-outlet café chain in Singapore. Revenue has been flat for two
> quarters, and the owner has to decide whether to renew the Marina Bay lease. She asks her
> assistant to send you the sales data. What arrives is a **raw till export**: one row per outlet,
> per day, per part of the day, straight out of the point-of-sale system, untouched.
>
> Nobody can answer the owner's question from this file yet. Today's job is to make it
> answerable — and to be able to say *why* every number in it can be trusted.

This is the first of three lessons on the same problem:

| Lesson | The question | What you do |
|---|---|---|
| **1.8 — today** | **Can I trust this data?** | clean one month of the raw export |
| 1.9 | What is the pattern? | 18 months, cleaned: time, joins, grouping |
| 1.10 | How do I make them act? | one chart, one slide, one decision |


### Setup

Import the libraries, then load the file we will use all session.


In [ ]:
# 👉 Load the two toolkits we need. `pd` and `np` are just short nicknames so we can
#    type `pd.something` instead of `pandas.something`. Run this cell first, every session.
import pandas as pd
import numpy as np


In [ ]:
# 👉 Load the dataset we will use all session: June 2025's till export from four cafés.
#    `read_csv` reads a comma-separated text file into a DataFrame -- a table with named columns.
#    pandas already treats an empty field, "NA" and "n/a" as missing.
raw = pd.read_csv("../data/cafe_june_raw.csv")

raw


### 🎬 Why this matters — before you trust a single number

Run the next three cells. The chain has **four** cafés and its busiest shift takes about \$1,000.


In [ ]:
# 👉 `.value_counts()` counts how many rows have each value. How many cafés do you count?
raw["outlet"].value_counts()


In [ ]:
# 👉 The same question of the daypart column. There are three parts to a trading day.
raw["daypart"].value_counts()


In [ ]:
# 👉 Sort the takings column and look at the two ends. `.dropna()` skips the blank cells,
#    because a sort cannot compare text with a blank -- which is itself a clue.
#    `.iloc[[0, -1]]` takes the first and last rows of the sorted result.
raw["revenue_raw"].dropna().sort_values().iloc[[0, -1]]


**Three problems, in three lines of output.**

1. **Twelve spellings for four cafés.** `Raffles Place`, `raffles place`, `RAFFLES PLACE`,
   `Raffles Pl.`… Group by outlet today and you get twelve cafés, four of which are the same shop.
2. **Nine labels for three dayparts** — `Morning`, `morning`, `AM`, and so on.
3. **The revenue column is not a number.** Sorted, the "smallest" value is `" 1,006.71 "` and the
   "largest" is `"S$94.41"`, because pandas is comparing them as **text**: a space sorts before a
   digit, and the letter `S` sorts after every digit. Sorted as text, \$98,000 loses to \$99.

Any average, chart or model built on this file is wrong before you start. Worse, none of it would
*look* wrong: it would produce numbers, with decimal places, and nobody in the meeting would know.

Part 1 is the routine that finds problems like these in about two minutes.


---

## Part 3: Data Transformation

**Learning outcome 3:** *Transform data through type conversion, string cleaning, and categorical
encoding.*

**Goal:** `clean` now has believable numbers, but you still cannot group by outlet (twelve
spellings), ask a date question (the date is text), or produce a per-outlet summary. That is this
section.

⏱️ ~35 min including Group Exercise 3


### 3.1: Transforming Data (Mapping)

A **mapping** is a lookup table: for each messy value, the value you want instead.


**On our dataset:** twelve outlet spellings for four cafés, and nine daypart labels for three
dayparts. A dictionary maps every messy spelling to the one we want.


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 See the mess in full before writing the mapping. `.unique()` lists distinct values.
clean["outlet"].unique()


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 A dictionary is a lookup table: {what_is_in_the_data: what_we_want}.
#    Writing it out by hand is not inelegant -- it is a record of a business decision.
outlet_map = {
    "Raffles Place": "Raffles Place",
    "raffles place": "Raffles Place",
    "RAFFLES PLACE": "Raffles Place",
    "Raffles Pl.": "Raffles Place",
    "Tampines Mall": "Tampines Mall",
    "tampines mall": "Tampines Mall",
    "Tampines  Mall": "Tampines Mall",
    "Marina Bay": "Marina Bay",
    "marina bay": "Marina Bay",
    "Marina Bay ": "Marina Bay",
    "Holland Village": "Holland Village",
    "Holland V": "Holland Village",
}

clean["outlet_name"] = clean["outlet"].map(outlet_map)

clean["outlet_name"].value_counts()


Four cafés. **A warning worth internalising:** if a thirteenth spelling appears next month,
`.map()` turns it into `NaN` silently. Always check afterwards.


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 The check that catches a spelling you did not know about.
#    If this is not 0, something in the outlet column was not in your dictionary.
clean["outlet_name"].isna().sum()


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 The daypart column has the same problem, and gets the same treatment.
daypart_map = {
    "Morning": "Morning", "morning": "Morning", "AM": "Morning",
    "Midday": "Midday", "midday": "Midday", "Lunch": "Midday",
    "Evening": "Evening", "evening": "Evening", "PM": "Evening",
}

clean["daypart"] = clean["daypart"].map(daypart_map)

clean["daypart"].value_counts()


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 And now the duplicate check from 2.2 can finally do its job: one row per outlet,
#    per date, per daypart. This is the check that was meaningless before the mapping.
clean.duplicated(subset=["date_text", "outlet_name", "daypart"]).sum()


Drills on smaller examples follow.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A tiny table of foods and weights, built from a dictionary of column-name: values.
foods = pd.DataFrame({
    "food": ["bacon", "pulled pork", "bacon", "pastrami", "corned beef", "bacon", "pastrami"],
    "ounces": [4, 3, 12, 6, 7.5, 8, 3],
})

foods


**Scenario:** add a column showing the animal each food came from.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A dictionary = a lookup table. Given a food (the key) it hands back an animal (the value).
meat_to_animal = {
    "bacon": "pig", "pulled pork": "pig", "pastrami": "cow", "corned beef": "cow",
}

meat_to_animal


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.map()` walks down the food column and swaps each value for its dictionary match.
#    Assign the result to a new column so the original stays intact.
foods["animal"] = foods["food"].map(meat_to_animal)

foods


You can also pass a **function** to `map()` for custom logic.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.map()` also accepts a function. `def` defines one: it takes an input x and returns
#    the looked-up value. Same result, more flexible.
def get_animal(x):
    return meat_to_animal[x]

foods["food"].map(get_animal)


**Replacing values:** `replace` is a specialised version of `map`, ideal for sentinel values.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Some systems use a fake number like -999 to mean 'nothing recorded' -- exactly like
#    the till in our own dataset.
sentinel_s = pd.Series([1., -999., 2., -999., -1000., 3.])

sentinel_s


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.replace()` swaps one value for another -- here, the fake code becomes a proper NaN.
sentinel_s.replace(-999, np.nan)


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Pass a list to replace several values with the same thing in one go.
sentinel_s.replace([-999, -1000], np.nan)


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Two lists of equal length: first list is what to find, second is what to put in its place.
sentinel_s.replace([-999, -1000], [np.nan, 0])


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 The clearest form: a dictionary of {old_value: new_value}. Same result, easier to read.
sentinel_s.replace({-999: np.nan, -1000: 0})


> **`.map()` vs `.replace()` — say which behaviour you want before you pick.**
> `.map()` needs an entry for *every* value and turns anything unlisted into `NaN`.
> `.replace()` changes only what you list and leaves everything else alone.
> For standardising a column with a known set of values, `.map()`'s strictness is a feature: it
> tells you when something new shows up.


### 3.2: Renaming Axis Labels

Changing row and column labels, using the same mapping idea.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A 3x4 table of the numbers 0-11. `np.arange(12)` makes 0..11 in a line and
#    `.reshape((3, 4))` folds it into 3 rows of 4.
labels = pd.DataFrame(
    np.arange(12).reshape((3, 4)),
    index=["Tampines", "Bishan", "Yishun"],
    columns=["one", "two", "three", "four"],
)

labels


Using `.map()` on the index:


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A function that shortens a label to its first 4 characters and upper-cases it.
#    `x[:4]` takes characters 0 to 3.
def shorten(x):
    return x[:4].upper()

labels.index.map(shorten)


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.index.map()` applies that function to every row label. Assigning back to `labels.index`
#    makes the change stick.
labels.index = labels.index.map(shorten)

labels


Using `.rename()` (which returns a copy by default):


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.rename()` is the tidier way to relabel. `str.title` and `str.upper` are ready-made
#    functions, passed without brackets because we want the function itself, not its result.
labels.rename(index=str.title, columns=str.upper)


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.rename()` also takes dictionaries when you only want to change specific labels.
labels.rename(index={"YISH": "NORTH"}, columns={"three": "peekaboo"})


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 On our own table: give the columns the names a report would use.
clean = clean.rename(columns={"staff_on_shift": "staff", "manager_email": "email"})

clean.columns


### 3.3: String Manipulation

Pandas has a special accessor `.str` that unlocks text methods for a whole column at once, and
handles missing values gracefully.


**On our dataset:** the `manager` column has stray spaces and inconsistent case, and we want the
domain out of each manager's email address.


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 Find the problem first. `repr()` shows the invisible characters, so spaces become visible.
[repr(v) for v in clean["manager"].unique()]


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 `.str` applies a text method to the whole column at once. Chain them left to right:
#    strip the spaces off the ENDS, squeeze any doubled spaces in the MIDDLE down to one,
#    then title-case what is left. s`\+` means "one or more whitespace characters".
clean["manager"] = (
    clean["manager"].str.strip().str.replace(r"\s+", " ", regex=True).str.title()
)

clean["manager"].unique()


**Why three methods and not two.** `.str.strip()` only removes spaces at the *ends*, so
`"Priya  Nair"` would have survived it — and one manager would have appeared twice in every summary,
with her shifts split between the two spellings. The `\s+` replacement is what catches the doubled
space inside the name. Always print `.unique()` after a text clean-up and count the values.

That chain would have fixed most of the outlet column too — but not `Raffles Pl.` or `Holland V`,
which are abbreviations rather than typos. **Text cleaning handles the mechanical mess; a mapping
handles the decisions.**


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 `.str.split("@")` cuts each email in two at the @ sign; `.str[1]` takes the second
#    piece -- the domain. Python counts from 0, so 1 is the second item.
clean["email"].str.split("@").str[1].value_counts(dropna=False)


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A drill on the same idea, small enough to see. Start from a dictionary of name: email,
#    then turn it into a Series so the names become the row labels.
emails = pd.Series({
    "Aisha": "aisha.rahman@dailygrind.sg",
    "Wei Ming": "weiming.tan@dailygrind.sg",
    "Priya": "priya.nair@dailygrind.sg",
    "Daniel": "daniel.lim@dailygrind.com.sg",
    "Unknown": np.nan,
})

emails


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.str` unlocks text methods for a whole column at once. Note the missing entry stays NaN
#    rather than raising an error -- which is exactly why `.str` exists.
emails.str.contains("dailygrind")


Note on data types: pandas has a dedicated text type (`string`) as well as the generic `object`.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Convert to the dedicated text type. Mostly the same, but missing values behave more
#    predictably: you get a proper <NA> rather than a float NaN inside a text column.
emails_str = emails.astype("string")

emails_str


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Same test on the text-typed column: now the answer is a proper True/False/<NA>.
emails_str.str.contains("dailygrind")


**Slicing:** you can treat the column like a Python string.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.str[:5]` slices every value to its first 5 characters.
emails.str[:5]


**Regex — one step at a time.** A *regular expression* is a pattern that describes the shape of
text rather than its exact content. Build it up in three steps.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `re` is Python's built-in regular-expression module -- pattern matching for text.
import re


**Step 1 — a pattern with no groups.** `.` means "any one character" and `+` means "one or more".
So `.+@.+` reads: something, an @ sign, something.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.str.contains` takes a regex, not just plain text. This asks: is there an @ sign
#    with at least one character on each side? A crude but useful "does this look like an email".
emails.str.contains(r".+@.+")


**Step 2 — one group.** Round brackets `( )` mark the part you want to *keep*.
`.str.extract` returns the captured part.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Read the pattern as: "an @ sign, then capture everything after it".
#    That capture is the domain.
emails.str.extract(r"@(.+)")


**Step 3 — three groups.** Same idea, three times, plus two new pieces of notation:

| Piece | Meaning |
|---|---|
| `[A-Z0-9._%+-]` | any one character from this set |
| `\.` | a literal dot (a bare `.` would mean "any character") |
| `+` | one or more of the thing before it |
| `flags=re.IGNORECASE` | treat upper and lower case as the same |


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A regex pattern with three bracketed groups: user, domain, suffix.
#    The `r"..."` prefix stops Python treating backslashes as escape characters.
pattern = r"([A-Z0-9._%+-]+)@([A-Z0-9.-]+)\.([A-Z]{2,4})"

emails.str.extract(pattern, flags=re.IGNORECASE)


> **Look at Daniel's row.** His address is `daniel.lim@dailygrind.com.sg`, and the pattern put
> `dailygrind.com` in the domain group and `sg` in the suffix. That is not a bug in the regex — it
> is the regex doing exactly what you asked, on data whose shape you had not thought about.
> Every regex you write needs testing against the awkward cases, not the tidy ones.


**Retrieving elements:** chain `.str` calls to get specific parts of a match.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.findall` returns a list of matches per row; `.str[0]` takes the first (and only) one,
#    which is a tuple of the three captured pieces.
matches = emails.str.findall(pattern, flags=re.IGNORECASE).str[0]

matches


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Pull item 1 out of each tuple. Python counts from 0, so 1 is the middle piece: the domain.
matches.str[1]


`extract` is usually the friendlier tool: it puts each group straight into its own column.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Name the groups and they become the column names -- much easier to read six months later.
emails.str.extract(r"(?P<user>[^@]+)@(?P<domain>.+)")


### 3.4: Categorical Data

Converting text columns to the `category` type saves memory and speeds things up. And `pd.cut`
turns continuous numbers into labelled bands, which is what most reports actually want.


**On our dataset:** the owner does not want 360 revenue figures. She wants to know how many shifts
were quiet, normal or busy.


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 `pd.cut` turns numbers into labelled bands. `bins` are the cut points and `labels` names them.
#    Read the bins as: 0-200, 200-500, 500-1200. bins are cut points, not ranges, so the first bin is 0 ≤ x < 200, the second is 200 ≤ x < 500, etc.
#   “Quiet” = 0-200, “Normal” = 200-500, “Busy” = 500-1200. Anything outside those ranges becomes NaN.
clean["shift_size"] = pd.cut(
    clean["revenue_sgd"],
    bins=[0, 200, 500, 1200],
    labels=["Quiet", "Normal", "Busy"],
)

clean["shift_size"].value_counts()


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 `.astype('category')` tells pandas this column has a small set of repeated values.
#    Same data, less memory -- and some operations get faster.
#    category is a special type of text column that is more efficient for repeated values, and it
#    also allows you to specify an order for the categories if you want to do comparisons.
#    The categories are ordered by the order they first appear in the data, not alphabetically.

clean["outlet_name"] = clean["outlet_name"].astype("category")

clean["outlet_name"].dtype


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 One-hot encoding: turn one text column into several 0/1 columns, one per daypart, so text data becomes numerical data.
#    This is what most machine-learning models need instead of text.

pd.get_dummies(clean["daypart"], prefix="is").head()


Drills on smaller examples follow.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A column of repeated drink names. `* 2` repeats the list, giving 8 rows.
drinks = pd.Series(["latte", "kopi", "latte", "latte"] * 2)

drinks


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Only two distinct drinks exist, even though there are 8 rows.
drinks.unique()


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 How many of each. This repetition is exactly what the `category` type optimises.
drinks.value_counts()


**Using the pandas `category` type:**


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Build a realistic little dataset. `rng` is a random-number generator with a fixed seed,
#    so everyone gets the same numbers.
rng = np.random.default_rng(seed=12345)
n = 8
drink_df = pd.DataFrame({
    "drink": ["latte", "kopi", "latte", "latte"] * 2,
    "count": rng.integers(3, 15, size=n),
    "weight": rng.uniform(0, 4, size=n).round(2),
})

drink_df


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `.astype('category')` asks pandas to store codes plus a lookup table instead of
#    repeating the text. Notice the dtype in the output.
drink_cat = drink_df["drink"].astype("category")

drink_cat


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Save the converted column back into the table so the change actually sticks.
drink_df["drink"] = drink_cat

drink_df.dtypes


**Binning data (`pd.cut`):** converting continuous numbers into categorical bands.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A plain Python list of shift takings -- continuous numbers we are about to group.
#    `pd.cut` turns numbers into labelled bands. `bins` are the cut points and `labels` names them.
#   if no labels are given, the output is a categorical column with the bin ranges as labels.

takings = [80, 145, 210, 260, 340, 480, 505, 610, 720, 880, 940, 1120]

bins = [0, 200, 500, 1200]

shifts = pd.cut(takings, bins)

shifts


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Which bin each value landed in, as a code number.
shifts.codes


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 The list of bin ranges that were created.
shifts.categories


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Count how many values fell into each bin -- an instant histogram in table form.
pd.Series(shifts).value_counts()


**Binning parameters:**


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `right=False` flips which edge is included: now 200 starts the middle bin
#    rather than ending the first one. Worth checking whenever a value sits exactly on a boundary.
pd.cut(takings, bins, right=False)


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Give the bins human-readable names instead of number ranges. The list of labels must
#    have exactly one fewer entry than the list of bin edges.
pd.cut(takings, bins, labels=["Quiet", "Normal", "Busy"])


If you pass an integer instead of edges, pandas computes equal-width bins for you.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Pass a plain number and pandas splits the range into that many equal-width bins.
#    `precision=2` just rounds the printed boundaries.
#   bin edge is exclusive on the left (a, b], so the minmum value would fall outside the first bin, the fix is to extend the range by 0.1% on the lower end, or use `include_lowest=True` to include the minimum value in the first bin.)
# range = 1120 - 80 = 1040, so each bin width = 1040 / 4 = 260
# pad = 1040 * 0.001 = 1.04
# left = 80 - 1.04 = 78.96

pd.cut(np.array(takings), 4, precision=2)


**Indicator / dummy variables:** converting a categorical column into binary columns.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 A small table with a text 'key' column, ready for encoding.
keys_df = pd.DataFrame({"key": list("bbacab"), "data1": range(6)})

keys_df


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 'One-hot encoding': one new 0/1 column per distinct value.
# get_dummies() is a convenient way to do this, and it automatically names the new columns after the distinct values in the original column. The result is a DataFrame with the same number of rows as the original, but with additional columns for each unique value in the 'key' column, filled with 0s and 1s indicating the presence of that value in each row.

pd.get_dummies(keys_df["key"])


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 `prefix=` puts a label in front, so you can tell where the columns came from
#    after you join them onto something else.

pd.get_dummies(keys_df["key"], prefix="key")


Recipe: combining `get_dummies` with `cut`.


In [ ]:
# ----- 🧪 DRILL (toy data) — practice only, not our dataset -----
# 👉 Combine the two ideas: `cut` groups the numbers into bins, then `get_dummies` turns
#    each bin into its own 0/1 column. Common as a final step before modelling.

values = np.random.default_rng(seed=12345).uniform(size=10)
edges = [0, 0.2, 0.4, 0.6, 0.8, 1.0]

pd.get_dummies(pd.cut(values, edges))


### 3.5: Type Conversion and Grouping

Two jobs left. `date_text` is still **text**, so we cannot ask a single date question of it. And we
still have no way to compare outlets — which is the question the owner actually asked.


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 Right now the dates are just strings. `.dtype` on one column confirms it: `object`.
clean["date_text"].dtype


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 `pd.to_datetime` parses text into real dates. `dayfirst=True` matters: "06/06/2025" is
#    unambiguous but "01/06/2025" is 1 June here and 6 January in the US, and pandas cannot know
#    which you meant. Tell it, or your report can be wrong by months with no error raised.
#.   '<M8[ns]' is the internal type for a date column, which is a 64-bit integer counting nanoseconds since 1970-01-01. The human-readable format is YYYY-MM-DD.

clean["date"] = pd.to_datetime(clean["date_text"], dayfirst=True)

clean["date"].dtype


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 `.dt` is the date accessor, the way `.str` is for text. Now these questions are possible:
print("first day:", clean["date"].min().date())
print("last day: ", clean["date"].max().date())

# 👉 Pull parts out of a date to group by later.
clean["weekday"] = clean["date"].dt.day_name()

clean[["date", "weekday"]].head()


**Grouping: split → apply → combine.** `groupby` splits the rows into groups, applies a
calculation to each group, and combines the answers into one table. It is the single most useful
summarising tool in pandas.


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 Split by outlet, then total the revenue within each group. Read it as a sentence:
#    "group by outlet name, take the revenue column, add it up".
clean.groupby("outlet_name", observed=True)["revenue_sgd"].sum().round(2)


In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 Several statistics at once with `.agg()`. Each line reads:
#    new_column_name = (which column, what calculation).
june_summary = clean.groupby("outlet_name", observed=True).agg(
    revenue=("revenue_sgd", "sum"),
    shifts=("revenue_sgd", "size"),
    avg_shift=("revenue_sgd", "mean"),
    tickets=("tickets", "sum"),
)
june_summary = june_summary.round(2)

june_summary


**This table is the point of the whole lesson.** It is only trustworthy because every number
behind it was checked: the sentinels are gone, the duplicate batch is gone, the mis-keyed
\$98,000 is gone, and the four cafés are four cafés rather than twelve.

Here is the same question asked of the raw file:


In [ ]:
# ===== 🏪 REAL DATASET (`raw`) — our coffee-shop data =====
# 👉 The same summary, on the untouched original. Twelve "outlets", and a total that is
#    more than 50% too high.
raw_numeric = raw.copy()
raw_numeric["revenue_sgd"] = pd.to_numeric(
    raw_numeric["revenue_raw"].str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce"
)

raw_numeric.groupby("outlet")["revenue_sgd"].sum().round(2)


> **Twelve rows instead of four, and no single row you could put in front of the owner.** This is
> what "the data was messy" costs in practice: not a slightly wrong answer, but a table nobody
> can use.


### 🛠️ Group Exercise 3 — Transformation (8 min)

Build a table with one row per `weekday` showing total revenue and the number of shifts, sorted busiest day first.

*Hint:* `.groupby("weekday").agg(...)`, naming the outputs in `.agg()` so you don't need to rename them afterwards.

---

## ✅ Sample Solution

Try the exercise yourself first — this is *a* solution, not *the* solution. If your code reaches the same answer a different way, it is right.

**Total revenue and number of shifts per weekday, busiest day first.**

In [ ]:
# ===== 🏪 REAL DATASET (`clean`) — our coffee-shop data =====
# 👉 One row per weekday, two numbers per row: the money total and how many shifts
#    went into it. Naming the outputs in `.agg()` saves renaming afterwards.
by_weekday = (
    clean.groupby("weekday")
    .agg(total_revenue=("revenue_sgd", "sum"), shifts=("revenue_sgd", "size"))
    .sort_values("total_revenue", ascending=False)
)

by_weekday


Always read the `shifts` column next to the money. A day with fewer shifts recorded will
look quieter even if its *per-shift* takings are the same — the count is what stops you drawing that wrong conclusion.

---

**⏭ Up next — Part 4: Reading and Writing Data (Self-Study).**

📂 Open `Part_4_reading_writing_data.ipynb` to continue.